# NeuroScan - InceptionV3
Two-phase transfer learning on Brain Tumor MRI dataset (4 classes).
Uses CUDA if available and compatible; falls back to CPU automatically.

In [ ]:
import subprocess, torch

r = subprocess.run(['nvidia-smi', '--query-gpu=name,compute_cap', '--format=csv,noheader'],
                   capture_output=True, text=True)
print('GPU        :', r.stdout.strip() or 'none')
print('PyTorch    :', torch.__version__)
print('CUDA avail :', torch.cuda.is_available())

device  = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
use_amp = device.type == 'cuda'

if device.type == 'cuda':
    try:
        torch.zeros(1, device=device)
        name  = torch.cuda.get_device_name(0)
        major, minor = torch.cuda.get_device_capability(0)
        print(f'GPU test   : OK ({name}, sm_{major}{minor})')
    except Exception as e:
        print(f'GPU failed : {type(e).__name__} - falling back to CPU')
        device  = torch.device('cpu')
        use_amp = False

print(f'\nUsing device : {device}  |  AMP : {use_amp}')

In [ ]:
import os

SEARCH_BASE = '/kaggle/input'

def find_dataset_root(base):
    # Walk recursively - handles any nesting depth Kaggle uses
    for root, dirs, files in os.walk(base):
        if 'Training' in dirs and 'Testing' in dirs:
            return root
    return None

print('Full /kaggle/input tree (up to depth 4):')
for root, dirs, files in os.walk(SEARCH_BASE):
    depth = root.replace(SEARCH_BASE, '').count(os.sep)
    if depth > 4:
        dirs[:] = []
        continue
    indent = '  ' * depth
    print(f'{indent}{os.path.basename(root) or "input"}/')

DATA_DIR = find_dataset_root(SEARCH_BASE)
if DATA_DIR is None:
    raise FileNotFoundError('Training/ + Testing/ not found under /kaggle/input - check dataset is attached')

print(f'\nData root  : {DATA_DIR}')
print(f'Training   : {os.path.isdir(DATA_DIR + "/Training")}')
print(f'Testing    : {os.path.isdir(DATA_DIR + "/Testing")}')

In [ ]:
import copy, time
import numpy as np
import pandas as pd
from torch import nn, optim
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models
from PIL import Image
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, classification_report
from tqdm import tqdm

CLASS_NAMES   = ['glioma', 'meningioma', 'notumor', 'pituitary']
NUM_CLASSES   = 4
IMG_SIZE      = 299
BATCH_SIZE    = 32 if device.type == 'cuda' else 16
EPOCHS_P1     = 10
EPOCHS_P2     = 10
LR_P1         = 1e-3
LR_P2         = 1e-5
NUM_WORKERS   = 2
OUTPUT_DIR    = '/kaggle/working/neuroscan/inception_v3'
MEAN          = [0.485, 0.456, 0.406]
STD           = [0.229, 0.224, 0.225]

print(f'Batch size : {BATCH_SIZE}  (GPU={device.type=="cuda"})')
print(f'Epochs     : {EPOCHS_P1} + {EPOCHS_P2}')
print(f'Output dir : {OUTPUT_DIR}')

In [ ]:
class BrainTumorDataset(Dataset):
    def __init__(self, root, transform=None):
        self.samples   = []
        self.label_map = {c: i for i, c in enumerate(CLASS_NAMES)}
        self.transform = transform
        for cls in CLASS_NAMES:
            cls_dir = os.path.join(root, cls)
            if not os.path.isdir(cls_dir):
                continue
            for fname in os.listdir(cls_dir):
                if fname.lower().endswith(('.jpg', '.jpeg', '.png')):
                    self.samples.append((os.path.join(cls_dir, fname), self.label_map[cls]))

    def __len__(self): return len(self.samples)

    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img = Image.open(path).convert('RGB')
        if self.transform: img = self.transform(img)
        return img, label


aug = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
basic = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

train_set = BrainTumorDataset(os.path.join(DATA_DIR, 'Training'), aug)
test_set  = BrainTumorDataset(os.path.join(DATA_DIR, 'Testing'),  basic)

train_loader = DataLoader(train_set, batch_size=BATCH_SIZE, shuffle=True,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))
test_loader  = DataLoader(test_set,  batch_size=BATCH_SIZE, shuffle=False,
                          num_workers=NUM_WORKERS, pin_memory=(device.type=='cuda'))

print(f'Train samples : {len(train_set)}')
print(f'Test  samples : {len(test_set)}')

In [ ]:
# Build InceptionV3
model = models.inception_v3(
    weights=models.Inception_V3_Weights.IMAGENET1K_V1, aux_logits=True)

model.AuxLogits.fc = nn.Linear(model.AuxLogits.fc.in_features, NUM_CLASSES)
model.fc = nn.Sequential(
    nn.Dropout(p=0.3),
    nn.Linear(model.fc.in_features, NUM_CLASSES),
)

# Phase 1: freeze backbone, train only new head
for param in model.parameters(): param.requires_grad = False
for param in model.fc.parameters():           param.requires_grad = True
for param in model.AuxLogits.fc.parameters(): param.requires_grad = True

model = model.to(device)

t = sum(p.numel() for p in model.parameters() if p.requires_grad)
n = sum(p.numel() for p in model.parameters())
print(f'Phase 1 trainable: {t:,} / {n:,} ({100*t/n:.1f}%)')

In [ ]:
def train_epoch(model, loader, criterion, optimizer):
    model.train()
    scaler = torch.amp.GradScaler('cuda', enabled=use_amp)
    loss_sum = correct = total = 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        optimizer.zero_grad()
        with torch.amp.autocast('cuda', enabled=use_amp):
            out, aux = model(imgs)
            loss = criterion(out, labels) + 0.4 * criterion(aux, labels)
        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


@torch.no_grad()
def eval_epoch(model, loader, criterion):
    model.eval()
    loss_sum = correct = total = 0
    for imgs, labels in tqdm(loader, leave=False):
        imgs, labels = imgs.to(device), labels.to(device)
        out  = model(imgs)
        loss = criterion(out, labels)
        loss_sum += loss.item() * imgs.size(0)
        correct  += (out.argmax(1) == labels).sum().item()
        total    += labels.size(0)
    return loss_sum / total, correct / total


def run_phase(phase_name, epochs, lr, unfreeze_fn=None):
    global model
    if unfreeze_fn: unfreeze_fn()
    t = sum(p.numel() for p in model.parameters() if p.requires_grad)
    n = sum(p.numel() for p in model.parameters())
    print(f'\n-- {phase_name} -- trainable: {t:,} / {n:,} ({100*t/n:.1f}%)')

    ckpt_dir = os.path.join(OUTPUT_DIR, 'checkpoints')
    os.makedirs(ckpt_dir, exist_ok=True)

    criterion = nn.CrossEntropyLoss()
    optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=lr)

    best_acc, best_state = 0.0, copy.deepcopy(model.state_dict())
    for epoch in range(1, epochs + 1):
        t0 = time.time()
        tr_loss, tr_acc = train_epoch(model, train_loader, criterion, optimizer)
        va_loss, va_acc = eval_epoch(model, test_loader, criterion)
        print(f'  Epoch {epoch:02d}/{epochs}  '
              f'train={tr_loss:.4f}/{tr_acc:.4f}  '
              f'val={va_loss:.4f}/{va_acc:.4f}  '
              f'({time.time()-t0:.1f}s)')
        if va_acc > best_acc:
            best_acc   = va_acc
            best_state = copy.deepcopy(model.state_dict())
            torch.save(best_state, os.path.join(ckpt_dir, f'{phase_name}_best.pt'))
        if epoch % 5 == 0:
            torch.save(model.state_dict(),
                       os.path.join(ckpt_dir, f'{phase_name}_epoch{epoch:02d}.pt'))

    model.load_state_dict(best_state)
    print(f'  Best val acc: {best_acc:.4f}')
    return best_acc

In [ ]:
# Phase 1 - feature extraction (head only)
run_phase('phase1', EPOCHS_P1, LR_P1)

In [ ]:
# Phase 2 - fine-tune Mixed_7* blocks + head
def unfreeze_p2():
    for param in model.parameters(): param.requires_grad = False
    for name, param in model.named_parameters():
        if 'Mixed_7' in name or name.startswith('fc') or 'AuxLogits' in name:
            param.requires_grad = True

run_phase('phase2', EPOCHS_P2, LR_P2, unfreeze_fn=unfreeze_p2)

In [ ]:
# Final evaluation
@torch.no_grad()
def full_eval(model, loader):
    model.eval()
    preds, labels = [], []
    for imgs, lbls in loader:
        preds.extend(model(imgs.to(device)).argmax(1).cpu().numpy())
        labels.extend(lbls.numpy())
    y_true, y_pred = np.array(labels), np.array(preds)
    print(f'Accuracy  : {accuracy_score(y_true, y_pred):.4f}')
    print(f'F1        : {f1_score(y_true, y_pred, average="weighted"):.4f}')
    print(f'Precision : {precision_score(y_true, y_pred, average="weighted", zero_division=0):.4f}')
    print(f'Recall    : {recall_score(y_true, y_pred, average="weighted"):.4f}')
    print()
    print(classification_report(y_true, y_pred, target_names=CLASS_NAMES))
    return float(accuracy_score(y_true, y_pred))

acc = full_eval(model, test_loader)

# Save final weights
out_dir = os.path.join(OUTPUT_DIR, 'outputs')
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, 'inception_v3_neuroscan.pt')
torch.save(model.state_dict(), out_path)
print(f'\nWeights saved: {out_path}')

In [ ]:
# Output summary
import os
print('Output files:')
for root, dirs, files in os.walk('/kaggle/working/neuroscan'):
    for f in files:
        path = os.path.join(root, f)
        mb = os.path.getsize(path) / 1e6
        print(f'  {path}  ({mb:.1f} MB)')